In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# read in all the words
words = open('../data/names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [3]:
#build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print (vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
# building the dataset
block_size = 3 # context length: how many characters do we take to predict the next one

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix] # crop and append
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

X_tr, Y_tr = build_dataset(words[:n1])      # 80%
X_dev, Y_dev = build_dataset(words[n1:n2])  # 10%
X_te, Y_te = build_dataset(words[n2:])      # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [5]:
# utility function for comparing manual gradients to PyTorch gradiends
def compare_gradients(s, d_t, t):
    exact = torch.all(d_t == t.grad).item()
    approx = torch.allclose(d_t, t.grad)
    max_difference = (d_t - t.grad).abs().max().item()

    print(f'{s:15s} | exact: {str(exact):5s} | approximate: {str(approx):5s} | max_difference: {max_difference}')

In [6]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # number of neurons in the hidden layer of MLP

g = torch.Generator().manual_seed(2147483647) # used for reproducibility
C = torch.randn((vocab_size, n_embd),             generator=g)

# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3) / ((n_embd * block_size) ** 0.5) # kaiming normalization (std)
b1 = torch.randn((n_hidden),                      generator=g) * 0.1 # not needed if we use batch normalization bias (bn_bias), using it now only for fun

# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn((vocab_size),                    generator=g) * 0.1

# BatchNorm parameters
bn_gain = torch.ones((1, n_hidden)) * 0.1 + 1.0
bn_bias = torch.zeros((1, n_hidden)) * 0.1

# Note: Initializing many of these parameter with e.g. all zeros could mas and incorrect implementation of the backward pass, so we are initializing them in non-standard way

parameters = [C, W1, b1, W2, b2, bn_gain, bn_bias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

4137


In [7]:
batch_size = 32

# minibatch construct
ix = torch.randint(0, X_tr.shape[0], (batch_size,), generator=g)
X_batch, Y_batch = X_tr[ix], Y_tr[ix] # batch X, Y

In [35]:
# forward pass, "chunked" into smaller steps that are possible to backward one at a time

emb = C[X_batch] # embed the characters into vectors
emb_cat = emb.view(emb.shape[0], -1) # concatenate the vectors

# linear layer 1
h_preact_bn = emb_cat @ W1 + b1 # hidden layer pre-activation

# BatchNorm layer
bn_mean_i = 1 / batch_size * h_preact_bn.sum(0, keepdim=True)
bn_diff = h_preact_bn - bn_mean_i # difference
bn_diff_sq = bn_diff ** 2 # difference squared
bn_var = 1 / (batch_size - 1) * (bn_diff_sq).sum(0, keepdim=True) # Bessel's correction (dividing by batch_size - 1, not batch_size)
bn_var_inv = (bn_var + 1e-5) ** -0.5
bn_raw = bn_diff * bn_var_inv
h_preact = bn_gain * bn_raw + bn_bias

# non-linearity
h = torch.tanh(h_preact) # hidden layer

# linear layer 2
logits = h @ W2 + b2 # output layer

# cross entropy loss (same as F.cross_entropy(logits, Y_batch))
logits_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logits_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1 # can't get backprop to be exact if using (1.0 / counts_sum)
probs = counts * counts_sum_inv
log_probs = probs.log()
loss = -log_probs[range(batch_size), Y_batch].mean()

# PyTorch backward pass
for p in parameters:
    p.grad = None

for t in [log_probs, probs, counts, counts_sum, counts_sum_inv, norm_logits, logits_maxes, logits, h, h_preact, bn_raw, bn_var_inv, bn_var, bn_diff_sq, bn_diff, h_preact_bn, bn_mean_i, emb_cat, emb]:
    t.retain_grad()

loss.backward()
loss

tensor(3.3482, grad_fn=<NegBackward0>)

In [39]:
# Exercise 1: backprop through the all variables as they are defined in the forward pass above, one by one
d_log_probs = torch.zeros_like(log_probs)
d_log_probs[range(batch_size), Y_batch] = -1.0 / batch_size
d_probs = (1.0 / probs) * d_log_probs
d_counts_sum_inv = (counts * d_probs).sum(1, keepdim=True)
d_counts = counts_sum_inv * d_probs
d_counts_sum = (-counts_sum ** -2) * d_counts_sum_inv
d_counts += torch.ones_like(counts) * d_counts_sum
d_norm_logits = (counts) * d_counts
d_logits = d_norm_logits.clone()
d_logits_maxes = (-d_logits).sum(1, keepdim=True)
d_logits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * d_logits_maxes
d_h = d_logits @ W2.T
d_W2 = h.T @ d_logits
d_b2 = d_logits.sum(0)
d_h_preact = (1.0 - h ** 2) * d_h
d_bn_gain = (bn_raw * d_h_preact).sum(0, keepdim=True)
d_bn_raw = bn_gain * d_h_preact
d_bn_bias = d_h_preact.sum(0, keepdim=True)
d_bn_diff = bn_var_inv * d_bn_raw
d_bn_var_inv = (bn_diff * d_bn_raw).sum(0, keepdim=True)
d_bn_var = (-0.5 * (bn_var + 1e-5) ** -1.5) * d_bn_var_inv
d_bn_diff_sq = (1.0 / (batch_size - 1)) * torch.ones_like(bn_diff_sq) * d_bn_var
d_bn_diff += (2 * bn_diff) * d_bn_diff_sq
d_h_preact_bn = d_bn_diff.clone()
d_bn_mean_i = (-d_bn_diff).sum(0)
d_h_preact_bn += 1.0 / batch_size * (torch.ones_like(h_preact_bn) * d_bn_mean_i)
d_emb_cat = d_h_preact_bn @ W1.T
d_W1 = emb_cat.T @ d_h_preact_bn
d_b1 = d_h_preact_bn.sum(0)
d_emb = d_emb_cat.view(emb.shape)
d_C = torch.zeros_like(C)
for k in range(X_batch.shape[0]):
    for j in range(X_batch.shape[1]):
        ix = X_batch[k, j]
        d_C[ix] += d_emb[k, j]

compare_gradients('log_probs', d_log_probs, log_probs)
compare_gradients('probs', d_probs, probs)
compare_gradients('counts_sum_inv', d_counts_sum_inv, counts_sum_inv)
compare_gradients('counts_sum', d_counts_sum, counts_sum)
compare_gradients('counts', d_counts, counts)
compare_gradients('norm_logits', d_norm_logits, norm_logits)
compare_gradients('logits_maxes', d_logits_maxes, logits_maxes)
compare_gradients('logits', d_logits, logits)
compare_gradients('h', d_h, h)
compare_gradients('W2', d_W2, W2)
compare_gradients('b2', d_b2, b2)
compare_gradients('h_preact', d_h_preact, h_preact)
compare_gradients('bn_gain', d_bn_gain, bn_gain)
compare_gradients('bn_bias', d_bn_bias, bn_bias)
compare_gradients('bn_raw', d_bn_raw, bn_raw)
compare_gradients('bn_var_inv', d_bn_var_inv, bn_var_inv)
compare_gradients('bn_var', d_bn_var, bn_var)
compare_gradients('bn_diff_sq', d_bn_diff_sq, bn_diff_sq)
compare_gradients('bn_diff', d_bn_diff, bn_diff)
compare_gradients('bn_mean_i', d_bn_mean_i, bn_mean_i)
compare_gradients('h_preact_bn', d_h_preact_bn, h_preact_bn)
compare_gradients('emb_cat', d_emb_cat, emb_cat)
compare_gradients('W1', d_W1, W1)
compare_gradients('b1', d_b1, b1)
compare_gradients('emb', d_emb, emb)
compare_gradients('C', d_C, C)

log_probs       | exact: True  | approximate: True  | max_difference: 0.0
probs           | exact: True  | approximate: True  | max_difference: 0.0
counts_sum_inv  | exact: True  | approximate: True  | max_difference: 0.0
counts_sum      | exact: True  | approximate: True  | max_difference: 0.0
counts          | exact: True  | approximate: True  | max_difference: 0.0
norm_logits     | exact: True  | approximate: True  | max_difference: 0.0
logits_maxes    | exact: True  | approximate: True  | max_difference: 0.0
logits          | exact: True  | approximate: True  | max_difference: 0.0
h               | exact: True  | approximate: True  | max_difference: 0.0
W2              | exact: True  | approximate: True  | max_difference: 0.0
b2              | exact: True  | approximate: True  | max_difference: 0.0
h_preact        | exact: True  | approximate: True  | max_difference: 0.0
bn_gain         | exact: True  | approximate: True  | max_difference: 0.0
bn_bias         | exact: True  | appro

In [41]:
# Exercise 2: backprop through cross_entropy all in one go

# forward pass here!

# Before:
# logits_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logits_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdim=True)
# counts_sum_inv = counts_sum ** -1 # can't get backprop to be exact if using (1.0 / counts_sum)
# probs = counts * counts_sum_inv
# log_probs = probs.log()
# loss = -log_probs[range(batch_size), Y_batch].mean()

# Now:
loss_fast = F.cross_entropy(logits, Y_batch)
print(loss_fast.item(), 'difference: ', (loss_fast - loss).item())

3.348198175430298 difference:  2.384185791015625e-07


In [42]:
# backward pass
d_logits = F.softmax(logits, 1)
d_logits[range(batch_size), Y_batch] -= 1
d_logits /= batch_size

compare_gradients('logits', d_logits, logits)

logits          | exact: False | approximate: True  | max_difference: 6.984919309616089e-09


In [ ]:
# Exercise 3: backprop through BatchNorm but all in one go

# forward pass here!

# Before:
# bn_mean_i = 1 / batch_size * h_preact_bn.sum(0, keepdim=True)
# bn_diff = h_preact_bn - bn_mean_i # difference
# bn_diff_sq = bn_diff ** 2 # difference squared
# bn_var = 1/(batch_size - 1) * (bn_diff_sq).sum(0, keepdim=True) # Bessel's correction (dividing by batch_size - 1, not batch_size)
# bn_var_inv = (bn_var + 1e-5) ** -0.5
# bn_raw = bn_diff * bn_var_inv
# h_preact = bn_gain * bn_raw + bn_bias

# Now:
h_preact_fast = bn_gain * (h_preact_bn - h_preact_bn.mean(0, keepdim=True)) / torch.sqrt(h_preact_bn.var(0, keepdim = True, unbiased=True))
print(h_preact_fast, 'max difference: ', (h_preact_fast - h_preact).abs().max())

In [43]:
# backward pass

# Before:
# d_bn_raw = bn_gain * d_h_preact
# d_bn_bias = d_h_preact.sum(0, keepdim=True)
# d_bn_diff = bn_var_inv * d_bn_raw
# d_bn_var_inv = (bn_diff * d_bn_raw).sum(0, keepdim=True)
# d_bn_var = (-0.5 * (bn_var + 1e-5) ** -1.5) * d_bn_var_inv
# d_bn_diff_sq = (1.0 / (batch_size - 1)) * torch.ones_like(bn_diff_sq) * d_bn_var
# d_bn_diff += (2 * bn_diff) * d_bn_diff_sq
# d_h_preact_bn = d_bn_diff.clone()
# d_bn_mean_i = (-d_bn_diff).sum(0)
# d_h_preact_bn += 1.0 / batch_size * (torch.ones_like(h_preact_bn) * d_bn_mean_i)

d_h_preact_bn = bn_gain * bn_var_inv / batch_size * (batch_size * d_h_preact - d_h_preact.sum(0) - batch_size / (batch_size - 1) * bn_raw * (d_h_preact * bn_raw).sum(0))

compare_gradients('h_preact_bn', d_h_preact_bn, h_preact_bn)

h_preact_bn     | exact: False | approximate: True  | max_difference: 9.313225746154785e-10


In [50]:
# Exercise 4: putting it all together! (train the MLP net with your own backward pass)

# initialization
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # number of neurons in the hidden layer of MLP

g = torch.Generator().manual_seed(2147483647) # used for reproducibility
C = torch.randn((vocab_size, n_embd),             generator=g)

# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3) / ((n_embd * block_size) ** 0.5) # kaiming normalization (std)
b1 = torch.randn((n_hidden),                      generator=g) * 0.1 # not needed if we use batch normalization bias (bn_bias), using it now only for fun

# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn((vocab_size),                    generator=g) * 0.1

# BatchNorm parameters
bn_gain = torch.ones((1, n_hidden)) * 0.1 + 1.0
bn_bias = torch.zeros((1, n_hidden)) * 0.1

# Note: Initializing many of these parameter with e.g. all zeros could mas and incorrect implementation of the backward pass, so we are initializing them in non-standard way

parameters = [C, W1, b1, W2, b2, bn_gain, bn_bias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
    p.requires_grad = True

# same optimization as before
max_steps = 200000
batch_size = 32
loss_i = []

# use this context manager for efficiency once the backward pass is written (TODO)
with torch.no_grad():

    for i in range(max_steps):

        # minibatch construct
        ix = torch.randint(0, X_tr.shape[0], (batch_size,), generator=g)
        X_batch, Y_batch = X_tr[ix], Y_tr[ix] # batch X, Y

        # forwards pass
        emb = C[X_batch] # embed the characters into vectors
        emb_cat = emb.view(emb.shape[0], -1) # concatenate the vectors

        # linear layer
        h_preact_bn = emb_cat @ W1 + b1 # hidden layer pre-activation

        # BatchNorm layer
        # --------------------------------------------------------------
        bn_mean = h_preact_bn.mean(0, keepdim=True)
        bn_var = h_preact_bn.var(0, keepdim=True, unbiased=True)
        bn_var_inv = (bn_var + 1e-5) ** -0.5
        bn_raw = (h_preact_bn - bn_mean) * bn_var_inv
        h_preact = bn_gain * bn_raw + bn_bias

        # non-linearity
        h = torch.tanh(h_preact) # hidden layer
        logits = h @ W2 + b2 # output layer
        loss = F.cross_entropy(logits, Y_batch) # loss function

        # backward pass
        for p in parameters:
            p.grad = None
        # loss.backward() # use this for correctness comparisons, delete it later!

        # manual backprop
        d_logits = F.softmax(logits, 1)
        d_logits[range(batch_size), Y_batch] -= 1
        d_logits /= batch_size

        # 2nd layer backprop
        d_h = d_logits @ W2.T
        d_W2 = h.T @ d_logits
        d_b2 = d_logits.sum(0)

        # tanh
        d_h_preact = (1.0 - h ** 2) * d_h

        # BatchNorm backprop
        d_bn_gain = (bn_raw * d_h_preact).sum(0, keepdim=True)
        d_bn_bias = d_h_preact.sum(0, keepdim=True)
        d_h_preact_bn = bn_gain * bn_var_inv / batch_size * (batch_size * d_h_preact - d_h_preact.sum(0) - batch_size / (batch_size - 1) * bn_raw * (d_h_preact * bn_raw).sum(0))

        # 1st layer
        d_emb_cat = d_h_preact_bn @ W1.T
        d_W1 = emb_cat.T @ d_h_preact_bn
        d_b1 = d_h_preact_bn.sum(0)

        # embedding
        d_emb = d_emb_cat.view(emb.shape)
        d_C = torch.zeros_like(C)
        for k in range(X_batch.shape[0]):
            for j in range(X_batch.shape[1]):
                ix = X_batch[k, j]
                d_C[ix] += d_emb[k, j]

        grads = [d_C, d_W1, d_b1, d_W2, d_b2, d_bn_gain, d_bn_bias]

        # update
        lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
        for p, grad in zip(parameters, grads):
            p.data += -lr * grad

        # track stats
        if i % 10000 == 0: # print every once in a while
            print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
        loss_i.append(loss.log10().item())

        # if i >= 100: # TODO: delete early breaking before training the full net
        #     break

12297
      0/ 200000: 3.8279
  10000/ 200000: 2.1609
  20000/ 200000: 2.4227
  30000/ 200000: 2.4362
  40000/ 200000: 2.0088
  50000/ 200000: 2.4084
  60000/ 200000: 2.4508
  70000/ 200000: 2.1090
  80000/ 200000: 2.3592
  90000/ 200000: 2.2353
 100000/ 200000: 1.9750
 110000/ 200000: 2.3438
 120000/ 200000: 2.0156
 130000/ 200000: 2.4772
 140000/ 200000: 2.3107
 150000/ 200000: 2.1108
 160000/ 200000: 1.9497
 170000/ 200000: 1.8004
 180000/ 200000: 2.0284
 190000/ 200000: 1.8848


In [49]:
# useful for checking the gradients
# for p, g in zip(parameters, grads):
#     compare_gradients(str(tuple(p.shape)), g, p)

In [51]:
# evaluate train and val loss
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
    x, y = {
        'train': (X_tr, Y_tr),
        'val': (X_dev, Y_dev),
        'test': (X_te, Y_te)
    }[split]
    emb = C[x] # (N, block_size, n_embed)
    emb_cat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embed)
    h_preact = emb_cat @ W1 + b1
    h_preact = bn_gain * (h_preact - bn_mean) * (bn_var + 1e-5) ** -0.5 + bn_bias # batch normalization
    h = torch.tanh(h_preact) # (N, n_hidden)
    logits = h @ W2 + b2 # (N, vocab_size)
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.102283239364624
val 2.1440281867980957


In [52]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
        # forward pass the neural net
        emb = C[torch.tensor([context])] # (1, block_size, n_embd)
        emb_cat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
        h_preact = emb_cat @ W1 + b1
        h_preact = bn_gain * (h_preact - bn_mean) * (bn_var + 1e-5) ** -0.5 + bn_bias
        h = torch.tanh(h_preact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)

        # sample from the distribution
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()

        # shift the context window and track samples
        context = context[1:] + [ix]
        out.append(ix)

        # if we sample the special '.' character, break
        if ix == 0:
            break
    
    print(''.join(itos[i] for i in out)) # decode and print the generated words

carlavela.
jhavif.
jarrige.
tyrence.
sae.
rahnen.
delyah.
jareei.
nellara.
chaily.
kaleigh.
ham.
joce.
quinthanlin.
anvi.
brion.
ell.
dearyn.
kai.
everluan.
